In [ ]:
#@title Cell 39.1 — Overview and validation boundary
from IPython.display import display, Markdown

display(Markdown("""
# NB39 — BMC: Model C data audit

**Purpose:** Prepare original observed MICs and independent Model C held-out predictions
for the BMC project (Bayesian Model C; without the chromosomal MIC score).

This notebook:
1. Finds the two original compressed input files, wherever they are under RefMIC.
2. Verifies 50,460 observations, 9,058 BioSamples, 26 antibiotics and five pathogen-out folds.
3. Matches observed MICs exactly with the original Notebook 26 interaction table.
4. Reports antibiotic-specific errors and severe MIC underprediction.
5. Exports validated files without refitting Model C or BMC.

**Important:** Model C outer-fold predictions are independent for evaluating *Model C*.
They are **not automatically an independent calibration-and-test dataset for BMC**:
using out-of-fold errors from the other outer folds to calibrate a test fold may
leak information because their Model C models were trained using that test fold.
A fully independent BMC evaluation needs training-only *inner* cross-fitted
Model C predictions for each outer fold (or a separate untouched test cohort).
No BMC calibration is performed in this audit.
"""))

In [ ]:
#@title Cell 39.2 — Locate original inputs anywhere on mounted Drive
import os
import json
import gzip
from pathlib import Path
import numpy as np
import pandas as pd

# Mount Google Drive (Colab only). Use an override for local Jupyter.
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get('BMC_PROJECT_ROOT', '/content/drive/MyDrive/RefMIC'))
if not ROOT.is_dir():
    raise FileNotFoundError(
        f'RefMIC project folder was not found: {ROOT}. '
        'If your RefMIC folder is elsewhere, set BMC_PROJECT_ROOT to that folder.'
    )

DRIVE_ROOT = Path('/content/drive') if Path('/content/drive').is_dir() else ROOT


def locate_original(filename):
    # Prefer an original stored in RefMIC, regardless of subfolder spelling.
    preferred = ROOT / '01_Original_Data' / filename
    if preferred.is_file():
        return preferred
    matches = sorted(p for p in ROOT.rglob(filename) if p.is_file())
    if not matches and DRIVE_ROOT != ROOT:
        # Includes MyDrive and any mounted shared drives.
        matches = sorted(p for p in DRIVE_ROOT.rglob(filename) if p.is_file())
    if not matches:
        raise FileNotFoundError(
            f'Could not find {filename} in RefMIC or under {DRIVE_ROOT}. '
            'Check the exact filename in Google Drive.'
        )
    if len(matches) > 1:
        raise ValueError(
            f'Multiple copies of {filename} were found. Please move the intended '
            'original into RefMIC/01_Original_Data or specify one exact path.\n'
            + '\n'.join(map(str, matches))
        )
    return matches[0]

PREDICTIONS_PATH = locate_original('26_nested_pathogen_out_predictions.csv.gz')
INTERACTIONS_PATH = locate_original('26_model_c_aligned_interactions.csv.gz')
OUTPUT_DIR = ROOT / '03_Results' / '39_BMC_ModelC_Data_Audit'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Held-out predictions:', PREDICTIONS_PATH)
print('Original MIC observations:', INTERACTIONS_PATH)
print('Results folder:', OUTPUT_DIR)


In [ ]:
#@title Cell 39.3 — Load and verify Model C held-out predictions
pred_all = pd.read_csv(PREDICTIONS_PATH, compression='gzip')
required = {'interaction_row','biosample','antibiotic','observed_log2_mic',
            'model','predicted_log2_mic','outer_fold'}
missing = sorted(required - set(pred_all.columns))
if missing:
    raise ValueError(f'Missing required columns in predictions: {missing}')
assert len(pred_all) == 100_920, f'Unexpected total prediction rows: {len(pred_all):,}'
assert set(pred_all['model']) == {'Model C','Model 3B baseline'}
assert not pred_all[list(required)].isna().any().any()
assert np.isfinite(pred_all['predicted_log2_mic']).all()
assert np.isfinite(pred_all['observed_log2_mic']).all()
assert set(pred_all['outer_fold'].astype(int)) == {1,2,3,4,5}
assert not pred_all.duplicated(['model','interaction_row']).any()

model_c = pred_all[pred_all['model'].eq('Model C')].copy()
model_3b = pred_all[pred_all['model'].eq('Model 3B baseline')].copy()
assert len(model_c) == len(model_3b) == 50_460
assert model_c['biosample'].nunique() == 9_058
assert model_c['antibiotic'].nunique() == 26
assert not model_c.duplicated(['biosample','antibiotic']).any()
assert (model_c.groupby('biosample')['outer_fold'].nunique() == 1).all(), 'One BioSample appears in multiple outer folds'
assert model_c['interaction_row'].nunique() == 50_460

model_c = model_c.sort_values('interaction_row').reset_index(drop=True)
model_c['residual_log2_mic'] = model_c['observed_log2_mic'] - model_c['predicted_log2_mic']
model_c['absolute_error_log2'] = model_c['residual_log2_mic'].abs()
model_c['underpredicted_ge_2_log2'] = model_c['residual_log2_mic'] >= 2.0
model_c['underpredicted_ge_4_log2'] = model_c['residual_log2_mic'] >= 4.0

print('Model C records:', format(len(model_c), ','))
print('BioSamples:', model_c['biosample'].nunique(), '| antibiotics:', model_c['antibiotic'].nunique(), '| outer folds:', model_c['outer_fold'].nunique())
print('Severe underprediction >=2 log2:', int(model_c['underpredicted_ge_2_log2'].sum()))
print('Severe underprediction >=4 log2:', int(model_c['underpredicted_ge_4_log2'].sum()))

In [ ]:
#@title Cell 39.4 — Verify against original MIC interactions and fold matching
original = pd.read_csv(INTERACTIONS_PATH, compression='gzip')
original_required = {'biosample','antibiotic','log2_mic'}
assert original_required.issubset(original.columns)
assert len(original) == 50_460
assert not original.duplicated(['biosample','antibiotic']).any()

matched = model_c.merge(original[['biosample','antibiotic','log2_mic']],
                        on=['biosample','antibiotic'], how='outer',
                        validate='one_to_one', indicator=True)
assert len(matched) == 50_460 and (matched['_merge'] == 'both').all()
assert np.array_equal(matched['observed_log2_mic'].to_numpy(), matched['log2_mic'].to_numpy()), 'Observed MICs differ from original!'

base_join = model_c[['interaction_row','biosample','antibiotic','outer_fold','observed_log2_mic']].merge(
    model_3b[['interaction_row','biosample','antibiotic','outer_fold','observed_log2_mic']],
    on='interaction_row', how='outer', validate='one_to_one',
    suffixes=('_c','_3b'),indicator=True)
assert len(base_join) == 50_460 and (base_join['_merge']=='both').all()
for col in ('biosample','antibiotic','outer_fold','observed_log2_mic'):
    assert (base_join[f'{col}_c']==base_join[f'{col}_3b']).all(), f'Disagreement with baseline on {col}'

print('PASS: Each Model C prediction matches its original observed MIC exactly.')
print('PASS: Model C and Model 3B share the same MIC observations and fold assignments.')
print('PASS: Every BioSample belongs to exactly one test fold.')

In [ ]:
#@title Cell 39.5 — Examine baseline errors for all 26 antibiotics

def summarise_group(group):
    residual = group['residual_log2_mic'].to_numpy(dtype=float)
    return pd.Series({
        'n_observed_mic': int(len(group)),
        'n_pathogens': int(group['biosample'].nunique()),
        'mean_residual_log2': float(residual.mean()),
        'MAE_log2': float(np.mean(np.abs(residual))),
        'RMSE_log2': float(np.sqrt(np.mean(residual**2))),
        'n_underprediction_ge_2_log2': int((residual>=2).sum()),
        'n_underprediction_ge_4_log2': int((residual>=4).sum()),
        'n_overprediction_ge_2_log2': int((residual<=-2).sum()),
    })

antibiotic_summary = (model_c.groupby('antibiotic',sort=True)
                      .apply(summarise_group, include_groups=False)
                      .reset_index())
for col in ('n_observed_mic','n_pathogens','n_underprediction_ge_2_log2',
            'n_underprediction_ge_4_log2','n_overprediction_ge_2_log2'):
    antibiotic_summary[col] = antibiotic_summary[col].astype(int)
assert antibiotic_summary['n_observed_mic'].sum()==50_460

print('Overall Model C held-out MAE:', round(model_c['absolute_error_log2'].mean(),4))
print('Overall Model C held-out RMSE:', round(np.sqrt(np.mean(model_c['residual_log2_mic']**2)),4))
print('Antibiotic-specific errors (sorted by number of severe underpredictions):')
display(antibiotic_summary.sort_values('n_underprediction_ge_2_log2',ascending=False).reset_index(drop=True))
print('Note: MIC >4 mg/L is NOT a common resistance threshold across 26 antibiotics; this audit uses prediction residuals instead.')

In [ ]:
#@title Cell 39.6 — Audit held-out folds and save validated outputs
fold_summary = (model_c.groupby('outer_fold',sort=True)
                .apply(summarise_group,include_groups=False)
                .reset_index())
for col in ('n_observed_mic','n_pathogens','n_underprediction_ge_2_log2',
            'n_underprediction_ge_4_log2','n_overprediction_ge_2_log2'):
    fold_summary[col]=fold_summary[col].astype(int)
assert int(fold_summary['n_observed_mic'].sum())==50_460
assert int(fold_summary['n_pathogens'].sum())==9_058
assert (fold_summary['n_observed_mic']==10_092).all()

out_pred = OUTPUT_DIR / '39_BMC_model_c_held_out_predictions.csv.gz'
out_drug = OUTPUT_DIR / '39_BMC_per_antibiotic_baseline_summary.csv'
out_fold = OUTPUT_DIR / '39_BMC_outer_fold_summary.csv'
out_extremes = OUTPUT_DIR / '39_BMC_severe_underpredictions.csv.gz'
out_manifest = OUTPUT_DIR / '39_BMC_input_audit.json'

model_c.to_csv(out_pred,index=False,compression='gzip')
antibiotic_summary.to_csv(out_drug,index=False)
fold_summary.to_csv(out_fold,index=False)
model_c[model_c['underpredicted_ge_2_log2']].to_csv(out_extremes,index=False,compression='gzip')
manifest = {
    'project':'BMC (Bayesian Model C)',
    'source_predictions_file':str(PREDICTIONS_PATH),
    'source_observed_mic_file':str(INTERACTIONS_PATH),
    'n_pathogens':int(model_c.biosample.nunique()),
    'n_antibiotics':int(model_c.antibiotic.nunique()),
    'n_observed_mic':len(model_c),
    'n_outer_folds':int(model_c.outer_fold.nunique()),
    'fold_assignments_verified':True,
    'observed_mic_matches_original':True,
    'strict_independent_bmc_calibration_available':False,
    'calibration_requirement': 'Generate training-only inner cross-fitted Model C predictions within each outer fold, or reserve a separate untouched independent test cohort.'
}
with open(out_manifest,'w',encoding='utf-8') as f:
    json.dump(manifest,f,indent=2)

display(fold_summary)
print('Saved 5 audit outputs in:',OUTPUT_DIR)
print('Original source files were not modified.')

In [ ]:
#@title Cell 39.7 — BMC validation design: required next input

display(Markdown("""
### NB39 conclusion

The uploaded Notebook 26 file contains **independent pathogen-out Model C predictions**,
not a leakage-free set of BMC calibration residuals for each evaluation fold.

For outer fold *k*, the proper procedure is:

1. Use only the other four outer folds as Model C training data.
2. **Inside that training data**, produce inner pathogen-out Model C predictions.
3. Fit BMC parameters to the resulting training-only residuals.
4. Apply BMC to Model C predictions of outer-fold *k* without using its observed MICs.
5. Repeat for five folds and compare BMC-N, BMC-T and BMC-AT.

**Do not:** fit BMC on all 50,460 observed errors and then describe the
results as independent validation. Also, simply excluding evaluation fold *k*
from the Notebook 26 held-out prediction table is not enough for strictly
independent BMC validation, because models generating the remaining folds'
predictions may themselves have been trained using pathogens in fold *k*.

Next: audit whether Notebook 26 saved the necessary **training-only inner
predictions** before deciding whether any Model C recalculation is required.
"""))